In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier


# Load train data only (no test usage in this notebook)
X_train = pd.read_csv(Path(r"data/X_train_selected.csv"))
y_train = pd.read_csv(Path(r"data/y_train.csv")).squeeze("columns")


In [ ]:
# Model
logreg_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

In [ ]:
# Model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

In [ ]:
# Model
xgb_model = XGBClassifier(
    n_estimators=450,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

In [ ]:
#Model 
cat_boost = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="Logloss",
    random_seed=42,
    verbose=False
)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}


def predict_top_fraction(y_prob, selection_rate):
    y_prob = np.asarray(y_prob)
    n_samples = y_prob.shape[0]
    n_selected = max(1, int(np.ceil(selection_rate * n_samples)))

    top_idx = np.argsort(y_prob)[::-1][:n_selected]
    y_pred = np.zeros(n_samples, dtype=int)
    y_pred[top_idx] = 1
    return y_pred


def threshold_for_selection_rate(y_prob, selection_rate):
    y_prob = np.asarray(y_prob)
    n_selected = max(1, int(np.ceil(selection_rate * y_prob.shape[0])))
    sorted_probs = np.sort(y_prob)[::-1]
    return float(sorted_probs[n_selected - 1])


def cumulative_gains_lift_curve(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    order = np.argsort(y_prob)[::-1]
    y_sorted = y_true[order]

    cumulative_positives = np.cumsum(y_sorted)
    total_positives = y_true.sum()
    population_fraction = np.arange(1, len(y_true) + 1) / len(y_true)

    if total_positives > 0:
        cumulative_gains = cumulative_positives / total_positives
    else:
        cumulative_gains = np.zeros_like(population_fraction)

    cumulative_lift = np.divide(
        cumulative_gains,
        population_fraction,
        out=np.zeros_like(population_fraction, dtype=float),
        where=population_fraction > 0,
    )

    return pd.DataFrame(
        {
            "population_fraction": population_fraction,
            "cumulative_gains": cumulative_gains,
            "cumulative_lift": cumulative_lift,
        }
    )


def evaluate_at_selection_rate(y_true, y_prob, selection_rate):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = predict_top_fraction(y_prob, selection_rate)

    prevalence = y_true.mean()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    realized_selection_rate = y_pred.mean()
    random_precision = prevalence
    random_recall_same_selection = realized_selection_rate

    return {
        "selection_rate": float(selection_rate),
        "realized_selection_rate": realized_selection_rate,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc,
        "random_precision": random_precision,
        "precision_lift_vs_random": precision / random_precision if random_precision > 0 else np.nan,
        "random_recall_same_selection": random_recall_same_selection,
        "recall_lift_vs_random_same_selection": (
            recall / random_recall_same_selection if random_recall_same_selection > 0 else np.nan
        ),
    }


def selection_rate_summary(y_true, y_prob, rates):
    rows = [evaluate_at_selection_rate(y_true, y_prob, rate) for rate in rates]
    return pd.DataFrame(rows)


def choose_elbow_selection_rate(summary_df, material_drop=0.15):
    df = summary_df.sort_values("selection_rate").copy()
    df["prev_lift"] = df["precision_lift_vs_random"].shift(1)
    df["lift_drop_pct"] = (df["prev_lift"] - df["precision_lift_vs_random"]) / df["prev_lift"]

    candidates = df[df["lift_drop_pct"] >= material_drop]
    if not candidates.empty:
        return float(candidates.iloc[0]["selection_rate"])

    return float(df.loc[df["precision_lift_vs_random"].idxmax(), "selection_rate"])


def plot_cumulative_gains_and_lift(curves_dict, title_prefix="Validation"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

    for model_name, curve_df in curves_dict.items():
        axes[0].plot(
            curve_df["population_fraction"] * 100,
            curve_df["cumulative_gains"],
            label=model_name,
        )
        axes[1].plot(
            curve_df["population_fraction"] * 100,
            curve_df["cumulative_lift"],
            label=model_name,
        )

    axes[0].plot([0, 100], [0, 1], linestyle="--", color="black", linewidth=1, label="Random")
    axes[0].set_title(f"{title_prefix} Cumulative Gains")
    axes[0].set_xlabel("Population selected (%)")
    axes[0].set_ylabel("Cumulative gains")
    axes[0].grid(alpha=0.3)

    axes[1].axhline(1.0, linestyle="--", color="black", linewidth=1, label="Random")
    axes[1].set_title(f"{title_prefix} Cumulative Lift")
    axes[1].set_xlabel("Population selected (%)")
    axes[1].set_ylabel("Cumulative lift")
    axes[1].grid(alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3)
    plt.tight_layout()
    plt.show()


In [ ]:
logreg_cv = cross_validate(
    logreg_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
logreg_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(logreg_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(logreg_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("Logistic Regression - Cross-Validation Results")
display(logreg_results)

In [ ]:
# CV
rf_cv = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
rf_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(rf_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(rf_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("Random Forest - Cross-Validation Results")
display(rf_results)

In [ ]:
# CV
cat_boost_cv = cross_validate(
    cat_boost,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
cat_boost_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(cat_boost_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(cat_boost_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("CatBoost - Cross-Validation Results")
display(cat_boost_results)

In [ ]:
# CV
xgb_cv = cross_validate(
    xgb_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
xgb_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(xgb_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(xgb_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("XGBoost - Cross-Validation Results")
display(xgb_results)

## Validation-based Operating-Point Selection (frozen deployment decision)


In [ ]:
validation_size = 0.20
selection_rate_grid = np.array([0.10, 0.20, 0.30, 0.40, 0.50])
material_lift_drop = 0.15

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=validation_size,
    random_state=42,
    stratify=y_train,
)

print(f"Fit split shape: {X_fit.shape}, Validation split shape: {X_val.shape}")

base_models = {
    "Logistic Regression": logreg_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "CatBoost": cat_boost,
}

validation_probabilities = {}
validation_curves = {}
validation_metrics_frames = []

for model_name, model in base_models.items():
    model_for_policy = clone(model)
    model_for_policy.fit(X_fit, y_fit)

    y_val_prob = model_for_policy.predict_proba(X_val)[:, 1]
    validation_probabilities[model_name] = y_val_prob

    validation_curves[model_name] = cumulative_gains_lift_curve(y_val, y_val_prob)

    summary_df = selection_rate_summary(y_val, y_val_prob, selection_rate_grid)
    summary_df.insert(0, "model", model_name)
    validation_metrics_frames.append(summary_df)

validation_metrics_df = pd.concat(validation_metrics_frames, ignore_index=True)

print("Validation metrics by selection rate:")
display(
    validation_metrics_df.sort_values(["selection_rate", "precision_lift_vs_random"], ascending=[True, False])
)

plot_cumulative_gains_and_lift(validation_curves, title_prefix="Validation")

policy_rows = []
for model_name in base_models.keys():
    model_summary = validation_metrics_df[validation_metrics_df["model"] == model_name].copy()
    selected_selection_rate = choose_elbow_selection_rate(model_summary, material_drop=material_lift_drop)

    y_val_prob = validation_probabilities[model_name]
    derived_threshold = threshold_for_selection_rate(y_val_prob, selected_selection_rate)

    selected_point = evaluate_at_selection_rate(y_val, y_val_prob, selected_selection_rate)

    policy_rows.append(
        {
            "model": model_name,
            "selected_selection_rate": selected_selection_rate,
            "derived_threshold": derived_threshold,
            "validation_realized_selection_rate": selected_point["realized_selection_rate"],
            "validation_precision": selected_point["precision"],
            "validation_recall": selected_point["recall"],
            "validation_f1": selected_point["f1"],
            "validation_lift_vs_random": selected_point["precision_lift_vs_random"],
            "selection_rule": f"first lift drop >= {material_lift_drop:.0%} or max lift fallback",
        }
    )

policy_df = pd.DataFrame(policy_rows).sort_values("model")

print("Frozen operating policy chosen on validation only:")
display(policy_df)


In [ ]:
from pathlib import Path

import joblib

# Fit models on full train data after policy selection is frozen on validation split
logreg_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
cat_boost.fit(X_train, y_train)

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(logreg_model, models_dir / "logreg_model.joblib")
joblib.dump(rf_model, models_dir / "rf_model.joblib")
joblib.dump(xgb_model, models_dir / "xgb_model.joblib")
joblib.dump(cat_boost, models_dir / "cat_boost.joblib")

policy_path = models_dir / "model_operating_points.csv"
validation_metrics_path = models_dir / "model_selection_rate_validation_metrics.csv"

policy_df.to_csv(policy_path, index=False)
validation_metrics_df.to_csv(validation_metrics_path, index=False)

print("Base models saved successfully.")
print(f"Frozen operating policy saved to {policy_path}")
print(f"Validation selection-rate metrics saved to {validation_metrics_path}")
